# Assignment 3: Hybrid Semantic Retrieval & Intelligence System (HSRIS)
### NLP Pipeline | Customer Support Tickets | PyTorch from Scratch

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
2
Tesla T4


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/waseemalastal/customer-support-ticket-dataset/customer_support_tickets.csv
/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.200d.txt
/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.50d.txt
/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt


In [4]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/waseemalastal/customer-support-ticket-dataset/customer_support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [5]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')

In [6]:
df.shape

(8469, 17)

## Part 1: Categorical Foundation — The Encoders

### 1a. Label Encoding — Ticket Priority

In [7]:
# Ordinal label encoding for Ticket Priority (from scratch — no sklearn)
priority_map = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
df["priority_encoded"] = df["Ticket Priority"].map(priority_map)
# Handle any unseen / NaN priorities gracefully
df["priority_encoded"] = df["priority_encoded"].fillna(-1).astype(int)
df[["Ticket Priority", "priority_encoded"]].head(10)

,Ticket Priority,priority_encoded
0,Critical,3
1,Critical,3
2,Low,0
3,Low,0
4,Low,0
5,Low,0
6,Critical,3
7,Critical,3
8,Low,0
9,Critical,3


### 1b. One-Hot Encoding — Ticket Channel

In [8]:
# One-Hot Encoding for Ticket Channel (from scratch — no sklearn)
known_channels = sorted(df["Ticket Channel"].dropna().unique().tolist())

def one_hot_channel(value, categories):
    """Return a binary vector; unknown category -> all-zero vector."""
    vec = [0] * len(categories)
    if value in categories:
        vec[categories.index(value)] = 1
    return vec

for ch in known_channels:
    col_name = "channel_" + ch.replace(" ", "_")
    df[col_name] = df["Ticket Channel"].apply(lambda x: 1 if x == ch else 0)

channel_cols = ["channel_" + ch.replace(" ", "_") for ch in known_channels]
print("Channels encoded:", known_channels)
df[["Ticket Channel"] + channel_cols].head(8)

Channels encoded: ['Chat', 'Email', 'Phone', 'Social media']


,Ticket Channel,channel_Chat,channel_Email,channel_Phone,channel_Social_media
0,Social media,0,0,0,1
1,Chat,1,0,0,0
2,Social media,0,0,0,1
3,Social media,0,0,0,1
4,Email,0,1,0,0
5,Social media,0,0,0,1
6,Social media,0,0,0,1
7,Social media,0,0,0,1


In [9]:
# Verify unseen category handling
test_unseen = one_hot_channel("Fax", known_channels)
print("Unseen channel Fax ->", test_unseen)  # should be all zeros

Unseen channel Fax -> [0, 0, 0, 0]


## Part 2: Sparse Representation — Keyword Retrieval

### 2a. Preprocessing & Display Text Construction

In [10]:
import re

def build_display_text(row):
    subject     = str(row.get("Ticket Subject", "")).strip()
    description = str(row.get("Ticket Description", ""))
    product     = str(row.get("Product Purchased", "the product")).strip()

    # 1. Replace template placeholder with actual product name
    description = description.replace("{product_purchased}", product)

    # 2. Remove ALL-CAPS tokens early (INFO, ERROR, WARN, PM ...)
    description = re.sub(r'\b[A-Z]{2,}\b', ' ', description)

    # 3. Remove log/thread fragments anywhere in string
    description = re.sub(
        r'(pm\s+)?client\s+thread\s+.*?(?=i\'ve|i\'m|my\s|the\s|we\s|$)',
        ' ', description, flags=re.IGNORECASE)

    # 4. Remove snake_case and PascalCase code tokens
    description = re.sub(r'\b[a-zA-Z]+(?:_[a-zA-Z]+)+\b', ' ', description)
    description = re.sub(r'\b[A-Z][a-z]+(?:[A-Z][a-z0-9]+)+\b', ' ', description)

    # 5. Remove timestamps e.g. 2023-06-01 12:15:36
    description = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}(\s+\d{1,2}:\d{2}(:\d{2})?)?\b', ' ', description)

    # 6. Remove standalone times e.g. 17:57:11
    description = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', description)

    # 7. Remove IP addresses
    description = re.sub(r'\b\d{1,3}(\.\d{1,3}){3}\b', ' ', description)

    # 8. Remove currency / prices e.g. $1,000.00
    description = re.sub(r'\$[\d,]+(\.\d+)?', ' ', description)

    # 9. Remove dimension patterns e.g. 5 1/2
    description = re.sub(r'\d+\s*\d*/\d+', ' ', description)

    # 10. Remove remaining standalone numbers
    description = re.sub(r'\b\d+(\.\d+)?\b', ' ', description)

    # 11. Remove URLs
    description = re.sub(r'http\S+|www\.\S+', ' ', description)

    # 12. Remove bullet / list markers
    description = re.sub(r'[*\u2022\-\u2013\u2014|]', ' ', description)

    # 13. Remove non-alphanumeric except basic punctuation
    description = re.sub(r'[^\w\s.,!?\']', ' ', description)

    # 14. Collapse whitespace
    description = re.sub(r'\s+', ' ', description).strip()

    # 15. Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', description)

    # 16. Filter out generic / broken / noisy sentences
    generic_patterns = [
        r"(?i)^i'?m having an issue with the .{0,60}$",
        r"(?i)^i'?m having an issue with the .{0,40}please assist",
        r"(?i)^i'?m facing a problem with my .{0,60}$",
        r"(?i)^please assist\.?$",
        r"(?i)^i need assistance as soon as possible",
        r"(?i)^if my product failed to reach",
        r"(?i)^i have a product .{0,40}please assist",
        r"(?i)^failed to \w+",
        r"(?i)^we will (not )?make any money",
        r"(?i)^if such payment is not possible",
        r"(?i)^try this",
        r"(?i)^client\s+thread",
        r"(?i)^to \w+ (package|value|store)",
    ]

    def is_meaningful(s):
        s = s.strip()
        if len(s) < 25:
            return False
        for pat in generic_patterns:
            if re.match(pat, s):
                return False
        return True

    good = [s.strip() for s in sentences if is_meaningful(s)]
    short_desc = " ".join(good[:2]) if good else description[:200]
    return subject, short_desc

df["display_subject"], df["display_description"] = zip(*df.apply(build_display_text, axis=1))
df["text"] = df["display_subject"] + " " + df["display_description"]

print("Sample display text:")
print(df[["display_subject", "display_description"]].head(5).to_string())

Sample display text:
            display_subject                                                                                                                                                                                                                        display_description
0             Product setup                                                                                                                                                        Your billing zip code is . We appreciate that you have requested a website address.
1  Peripheral compatibility                                                                                                                                                        If you need to change an existing product. If The issue I'm facing is intermittent.
2           Network problem                                                                                                                                               The Dell is not turn

### 2b. Text Cleaning & Custom Tokenizer

In [11]:
def clean_text(t):
    """Clean text for tokenization / TF-IDF / embedding (no sklearn)."""
    t = str(t)

    # Remove template placeholders
    t = re.sub(r'\{[^}]*\}', ' ', t)

    # Remove ALL-CAPS tokens
    t = re.sub(r'\b[A-Z]{2,}\b', ' ', t)

    # Remove log/thread fragments
    t = re.sub(
        r'(pm\s+)?client\s+thread\s+.*?(?=i\'ve|i\'m|my\s|the\s|we\s|$)',
        ' ', t, flags=re.IGNORECASE)

    # Remove snake_case and PascalCase code tokens
    t = re.sub(r'\b[a-zA-Z]+(?:_[a-zA-Z]+)+\b', ' ', t)
    t = re.sub(r'\b[A-Z][a-z]+(?:[A-Z][a-z0-9]+)+\b', ' ', t)

    # Remove timestamps, IPs, prices, numbers
    t = re.sub(r'\b\d{1,4}[/-]\d{1,2}[/-]\d{1,4}(\s+\d{1,2}:\d{2}(:\d{2})?)?\b', ' ', t)
    t = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', t)
    t = re.sub(r'\b\d{1,3}(\.\d{1,3}){3}\b', ' ', t)
    t = re.sub(r'\$[\d,]+(\.\d+)?', ' ', t)
    t = re.sub(r'\d+\s*\d*/\d+', ' ', t)
    t = re.sub(r'\b\d+(\.\d+)?\b', ' ', t)

    # Remove URLs, bullets, special chars
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'[*\u2022\-\u2013\u2014|\\]', ' ', t)
    t = re.sub(r'[^a-zA-Z\s]', ' ', t)

    # Remove very short tokens (1-2 chars)
    t = re.sub(r'\b[a-zA-Z]{1,2}\b', ' ', t)

    # Lowercase and collapse whitespace
    t = re.sub(r'\s+', ' ', t).strip().lower()
    return t

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head(3)

,text,clean_text
0,Product setup Your billing zip code is . We ap...,product setup your billing zip code appreciate...
1,Peripheral compatibility If you need to change...,peripheral compatibility you need change exist...
2,Network problem The Dell is not turning on. It...,network problem the dell not turning was worki...


In [12]:
# Custom regex-based tokenizer (no sklearn)
def tokenize(text):
    return re.findall(r'\b[a-z]+\b', text)

df["tokens"] = df["clean_text"].apply(tokenize)
df["tokens"].head()

0    [product, setup, your, billing, zip, code, app...
1    [peripheral, compatibility, you, need, change,...
2    [network, problem, the, dell, not, turning, wa...
3    [account, access, you, have, problem, you, int...
4    [data, loss, note, the, seller, not, responsib...
Name: tokens, dtype: object

### 2c. Count Vectorizer (Bag of Words) — Top 5,000 Tokens

In [13]:
from collections import Counter
import numpy as np

word_counter = Counter()
for tokens in df["tokens"]:
    word_counter.update(tokens)

print("Total unique words:", len(word_counter))

vocab_size = 5000
vocab    = [w for w, _ in word_counter.most_common(vocab_size)]
word2idx = {w: i for i, w in enumerate(vocab)}

print("Vocabulary size:", len(vocab))
print("Sample vocab:", vocab[:10])

Total unique words: 4686
Vocabulary size: 4686
Sample vocab: ['the', 'issue', 'and', 'product', 'you', 'this', 'but', 'problem', 'for', 'not']


In [14]:
def bow_vector(tokens):
    """Bag-of-words count vector (from scratch)."""
    vec = np.zeros(vocab_size)
    for w in tokens:
        if w in word2idx:
            vec[word2idx[w]] += 1
    return vec

df["bow"] = df["tokens"].apply(bow_vector)
print("BoW vector shape:", df["bow"].iloc[0].shape)
df["bow"].iloc[0][:20]

BoW vector shape: (5000,)


array([0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 1., 0.])

### 2d. N-Gram Generator — Bigrams & Trigrams

In [15]:
def generate_ngrams(tokens, n):
    """Sliding window n-gram generator."""
    return ["_".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

print("Bigrams :", generate_ngrams(["payment", "not", "working"], 2))
print("Trigrams:", generate_ngrams(["payment", "not", "working", "today"], 3))

Bigrams : ['payment_not', 'not_working']
Trigrams: ['payment_not_working', 'not_working_today']


In [16]:
df["bigrams"]  = df["tokens"].apply(lambda x: generate_ngrams(x, 2))
df["trigrams"] = df["tokens"].apply(lambda x: generate_ngrams(x, 3))
df[["tokens", "bigrams", "trigrams"]].head(2)

,tokens,bigrams,trigrams
0,"[product, setup, your, billing, zip, code, app...","[product_setup, setup_your, your_billing, bill...","[product_setup_your, setup_your_billing, your_..."
1,"[peripheral, compatibility, you, need, change,...","[peripheral_compatibility, compatibility_you, ...","[peripheral_compatibility_you, compatibility_y..."


### 2e. TF-IDF Implementation (from scratch)

In [17]:
import math

doc_freq = Counter()
for tokens in df["tokens"]:
    for w in set(tokens):
        doc_freq[w] += 1

N = len(df)
print("Total documents:", N)

# Smooth IDF (sklearn-style): log((N+1)/(df+1)) + 1
idf = {}
for w in vocab:
    dfreq = doc_freq.get(w, 1)
    idf[w] = math.log((N + 1) / (dfreq + 1)) + 1

print("Sample IDF scores:", {w: round(idf[w], 3) for w in vocab[:5]})

Total documents: 8469
Sample IDF scores: {'the': 1.221, 'issue': 1.74, 'and': 1.854, 'product': 2.082, 'you': 2.494}


In [18]:
def compute_tfidf(tokens):
    """Compute TF-IDF vector for a token list (from scratch)."""
    if len(tokens) == 0:
        return np.zeros(vocab_size)
    tf = {}
    for w in tokens:
        tf[w] = tf.get(w, 0) + 1
    total = len(tokens)
    for w in tf:
        tf[w] /= total
    vec = np.zeros(vocab_size)
    for w, freq in tf.items():
        if w in word2idx:
            vec[word2idx[w]] = freq * idf.get(w, 0)
    return vec

df["tfidf"] = df["tokens"].apply(compute_tfidf)
print("TF-IDF vector length:", len(df["tfidf"][0]))
df["tfidf"].iloc[0][:20]

TF-IDF vector length: 5000


array([0.        , 0.        , 0.        , 0.16017065, 0.19185773,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.21023595, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.2432191 , 0.        ])

### 2f. Sparse Tensor Storage (RAM-safe)

In [19]:
# Stack TF-IDF into matrix then convert to sparse tensor (saves Kaggle RAM)
tfidf_matrix = np.vstack(df["tfidf"].values)
print("Dense TF-IDF matrix shape:", tfidf_matrix.shape)

tfidf_tensor = torch.tensor(tfidf_matrix, dtype=torch.float32).to(device)
tfidf_sparse = tfidf_tensor.to_sparse()

print("Dense tensor shape :", tfidf_tensor.shape)
print("Sparse tensor shape:", tfidf_sparse.shape)
print("Non-zero elements  :", tfidf_sparse._nnz())

Dense TF-IDF matrix shape: (8469, 5000)
Dense tensor shape : torch.Size([8469, 5000])
Sparse tensor shape: torch.Size([8469, 5000])
Non-zero elements  : 160976


## Part 3: Dense Semantic Layer — Neural Embeddings

### 3a. Load GloVe 200-d Vectors

In [20]:
glove = {}
glove_path = "/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.200d.txt"

with open(glove_path, "r", encoding="utf8") as f:
    for line in f:
        parts  = line.split()
        word   = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove[word] = vector

EMBED_DIM = 200
print("Total GloVe words:", len(glove))
print("Vector size:", len(glove["computer"]))

Total GloVe words: 400000
Vector size: 200


### 3b. Load GloVe Weights into torch.nn.Embedding

In [21]:
# Build embedding matrix aligned with our vocabulary
# OOV tokens -> zero vector (UNK strategy)
unk_vector = np.zeros(EMBED_DIM, dtype=np.float32)

embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype=np.float32)
oov_count = 0

for w, idx in word2idx.items():
    vec = glove.get(w, None)
    if vec is not None:
        embedding_matrix[idx] = vec
    else:
        oov_count += 1
        embedding_matrix[idx] = unk_vector

print(f"OOV tokens: {oov_count} / {vocab_size} ({100*oov_count/vocab_size:.1f}%)")

# Load into torch.nn.Embedding with frozen pretrained weights
embedding_layer = torch.nn.Embedding(vocab_size, EMBED_DIM)
embedding_layer.weight = torch.nn.Parameter(
    torch.tensor(embedding_matrix, dtype=torch.float32),
    requires_grad=False
)
embedding_layer = embedding_layer.to(device)
print("Embedding layer:", embedding_layer)

OOV tokens: 389 / 5000 (7.8%)
Embedding layer: Embedding(5000, 200)


### 3c. TF-IDF Weighted Sentence Embedding (prevents semantic dilution)

In [22]:
def sentence_embedding(tokens, tfidf_vec):
    """
    TF-IDF weighted average of GloVe vectors.
    Rare technical words get higher weight than common words,
    preventing semantic dilution. OOV tokens -> zero vector.
    """
    sent_vec     = np.zeros(EMBED_DIM)
    total_weight = 0.0
    for w in tokens:
        if w in word2idx:
            weight       = tfidf_vec[word2idx[w]]
            word_vec     = glove.get(w, unk_vector)
            sent_vec    += weight * word_vec
            total_weight += weight
    if total_weight > 0:
        sent_vec /= total_weight
    return sent_vec

df["semantic_vec"] = df.apply(
    lambda row: sentence_embedding(row["tokens"], row["tfidf"]), axis=1)

print("Semantic vector length:", len(df["semantic_vec"][0]))
df["semantic_vec"].iloc[0][:10]

Semantic vector length: 200


array([ 0.26550896,  0.25889506,  0.33168345, -0.08756673,  0.14945391,
       -0.09377598, -0.51290619, -0.04446621, -0.02457142,  0.11429735])

In [23]:
semantic_matrix = np.vstack(df["semantic_vec"].values)
semantic_tensor = torch.tensor(semantic_matrix, dtype=torch.float32).to(device)
print("Semantic tensor shape:", semantic_tensor.shape)

Semantic tensor shape: torch.Size([8469, 200])


## Implementation Tasks

### Task 1 & 2: Hybrid Search Pipeline
`FinalScore = α·(TF-IDF Score) + (1−α)·(GloVe Score)`  — α = 0.4

In [24]:
import torch.nn.functional as F

def query_embedding(query):
    """Convert raw query string to GloVe sentence embedding."""
    q      = clean_text(query)
    tokens = tokenize(q)
    tv     = compute_tfidf(tokens)
    return sentence_embedding(tokens, tv)

def query_tfidf_vector(query):
    """Convert raw query string to TF-IDF vector."""
    q      = clean_text(query)
    tokens = tokenize(q)
    return compute_tfidf(tokens)

def semantic_search(query_vec, sem_tensor):
    """Cosine similarity using GloVe embeddings on GPU."""
    q = torch.tensor(query_vec, dtype=torch.float32).to(device)
    return F.cosine_similarity(q.unsqueeze(0), sem_tensor)

def tfidf_search(query_tfidf_vec):
    """Cosine similarity using TF-IDF on GPU."""
    q = torch.tensor(query_tfidf_vec, dtype=torch.float32).to(device)
    return F.cosine_similarity(q.unsqueeze(0), tfidf_tensor)

def hybrid_search(query, alpha=0.4):
    """
    FinalScore = alpha * TF-IDF_Score + (1 - alpha) * GloVe_Score
    alpha = 0.4 as specified in assignment.
    """
    sem_scores   = semantic_search(query_embedding(query), semantic_tensor)
    tfidf_scores = tfidf_search(query_tfidf_vector(query))
    return alpha * tfidf_scores + (1 - alpha) * sem_scores

print("Search functions defined")

Search functions defined


In [25]:
def display_results(query, top_indices, top_scores, label="Hybrid"):
    """Pretty-print retrieval results."""
    print(f"Top 5 {label} results for query: '{query}'")
    print("=" * 60)
    for rank, (idx, score) in enumerate(
            zip(top_indices.cpu().numpy(), top_scores.cpu().numpy()), 1):
        row = df.iloc[idx]
        print(f"Rank {rank}  |  Score: {score:.4f}  |  Type: {row['Ticket Type']}")
        print(f"  Subject    : {row['display_subject']}")
        print(f"  Description: {row['display_description']}")
        print()

### Semantic-Only Search Test

In [26]:
query      = "payment failed"
sem_scores = semantic_search(query_embedding(query), semantic_tensor)
top5       = torch.topk(sem_scores, 5)
display_results(query, top5.indices, top5.values, label="Semantic (GloVe)")

Top 5 Semantic (GloVe) results for query: 'payment failed'
Rank 1  |  Score: 0.7831  |  Type: Refund request
  Subject    : Product setup
  Description: I have to add my new payment. My payment will be placed soon...

Rank 2  |  Score: 0.7445  |  Type: Billing inquiry
  Subject    : Display issue
  Description: Please provide payment options through Paypal, Credit card, , Sender Mail etc. Please use the Payment method indicated above.

Rank 3  |  Score: 0.7293  |  Type: Product inquiry
  Subject    : Data loss
  Description: I'm having an issue with the Nintendo Switch. Please assist. We will make any money without payment of the payment. If such payment is not possible at the time listed on the product, we will contact y

Rank 4  |  Score: 0.7223  |  Type: Product inquiry
  Subject    : Delivery problem
  Description: When you checkout, your payment must be deposited in a separate account on the payment processor. Returns and exchange restrictions I've performed a factory reset on my 

### TF-IDF-Only Search Test

In [27]:
tfidf_scores = tfidf_search(query_tfidf_vector(query))
top5_tfidf   = torch.topk(tfidf_scores, 5)
display_results(query, top5_tfidf.indices, top5_tfidf.values, label="TF-IDF (Keyword)")

Top 5 TF-IDF (Keyword) results for query: 'payment failed'
Rank 1  |  Score: 0.5201  |  Type: Refund request
  Subject    : Product setup
  Description: I have to add my new payment. My payment will be placed soon...

Rank 2  |  Score: 0.4626  |  Type: Technical issue
  Subject    : Payment issue
  Description: Please help me through it.

Rank 3  |  Score: 0.4553  |  Type: Product inquiry
  Subject    : Data loss
  Description: I'm having an issue with the Nintendo Switch. Please assist. We will make any money without payment of the payment. If such payment is not possible at the time listed on the product, we will contact y

Rank 4  |  Score: 0.3232  |  Type: Technical issue
  Subject    : Peripheral compatibility
  Description: I am an Interactive Payment Network , and in order to get a payment in your country, must visit I'm experiencing this issue on multiple devices of the same model, so it seems to be a widespread problem.

Rank 5  |  Score: 0.3200  |  Type: Cancellation request


### Hybrid Search Test (α = 0.4)

In [28]:
query  = "payment failed"
scores = hybrid_search(query, alpha=0.4)
top5   = torch.topk(scores, 5)
display_results(query, top5.indices, top5.values, label="Hybrid (alpha=0.4)")

Top 5 Hybrid (alpha=0.4) results for query: 'payment failed'
Rank 1  |  Score: 0.6779  |  Type: Refund request
  Subject    : Product setup
  Description: I have to add my new payment. My payment will be placed soon...

Rank 2  |  Score: 0.6197  |  Type: Product inquiry
  Subject    : Data loss
  Description: I'm having an issue with the Nintendo Switch. Please assist. We will make any money without payment of the payment. If such payment is not possible at the time listed on the product, we will contact y

Rank 3  |  Score: 0.5787  |  Type: Technical issue
  Subject    : Payment issue
  Description: Please help me through it.

Rank 4  |  Score: 0.5569  |  Type: Billing inquiry
  Subject    : Display issue
  Description: Please provide payment options through Paypal, Credit card, , Sender Mail etc. Please use the Payment method indicated above.

Rank 5  |  Score: 0.5358  |  Type: Product inquiry
  Subject    : Delivery problem
  Description: When you checkout, your payment must be depo